# BRMA adaptive-attack study (CIC-IDS2017, 12 seeds, one run)
Tests whether BRMA survives attackers who know the defence, and adds a self-sufficient high-loss baseline.

* **S3H / S5 (selective attack):** each attacker scores its own DoS Hulk records with its own copy of BRMA and flips only the most benign-looking ones (3 attackers 90%, 5 attackers 50%).
* **M3H / M5 (camouflage attack):** usual flip, plus the flipped records' features are moved halfway towards the attacker's benign traffic.
* **K3H / K5 / C:** the standard attacks and clean data, for reference.
* Methods: NONE, BRMA, RAND, LOSS (BRMA's budget), **LOSS5 (fixed 5%, no BRMA)**, ORACLE, DC, MEDIAN.
* Pre-declared tests (Holm over four): T7 BRMA < NONE at S3H; T8 BRMA < NONE at M3H; T9 BRMA < LOSS5 at K3H; T10 BRMA < LOSS5 at S3H.

**How to run:** Settings: **Internet ON**, Accelerator **None (CPU)**. Then **Save Version -> Save & Run All (Commit)**.
About 6-8 hours for all 12 seeds (7 arms x 8 methods = 672 runs). When finished, download `brma-adaptive.zip` from the Output tab.

In [ ]:
# 1. Config and helpers
import os, sys, subprocess, time, json, hashlib, shutil, platform
W = "/kaggle/working"
R = f"{W}/repo"                      # clone of the project repository
VENV = f"{W}/venv312"
VPY = f"{VENV}/bin/python"
STATE = f"{W}/state"
OUT1, OUT2, OUT3 = (f"{W}/outputs/part{i}" for i in (1, 2, 3))   # results are kept separate per part
for d in (STATE, OUT1, OUT2, OUT3): os.makedirs(d, exist_ok=True)
REPO_URL = "https://github.com/ahmedbilalazhar/cl-ids-adversarial.git"
EXPECTED_COMMIT = "571b5f8"          # commit the plan was written against
DATA_URL = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip"
DATA_SHA256 = "97fdb91d339e2d8cf5627f981b831e5e7e400b981c58181c451a38fd03c48883"
DATA_BYTES = 343549013
DEST = "kaggle-reproduction"         # fresh output name: existing result folders are immutable

def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)

def run(cmd, cwd=R, logfile=None, check=True):
    """Run a command, stream its output live, stop the notebook on failure."""
    log("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(logfile, "a") if logfile else None
    for line in p.stdout:
        print(line, end="")
        if f: f.write(line)
    p.wait()
    if f: f.close()
    log(f"exit {p.returncode} after {time.time()-t0:.0f}s")
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
    return h.hexdigest()

def done(name): return os.path.exists(f"{STATE}/{name}.done")
def mark(name): open(f"{STATE}/{name}.done", "w").write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))

In [ ]:
# 2. Environment report and repository clone
log("Python", sys.version.split()[0], "|", platform.platform())
log("CPU cores", os.cpu_count())
try:
    import psutil; log("RAM GB", round(psutil.virtual_memory().total / 1e9, 1))
except Exception as e: log("psutil unavailable", e)
log("Free disk GB", round(shutil.disk_usage(W).free / 1e9, 1))
if not os.path.isdir(f"{R}/.git"):
    run(f"git clone {REPO_URL} {R}", cwd=W)
head = subprocess.run("git log --oneline -1", shell=True, cwd=R, capture_output=True, text=True).stdout.strip()
log("Repository HEAD:", head)
if not head.startswith(EXPECTED_COMMIT):
    print("WARNING: HEAD differs from the commit this plan was written for. Review the new commits before continuing.")

In [ ]:
# 3. Python 3.12 environment with the repository's exact pins (reuses an existing valid environment)
WANT = ["2.3.5", "3.0.1", "1.18.1", "1.9.1", "2.7.0+cpu"]
def venv_versions():
    if not os.path.exists(VPY): return None
    r = subprocess.run([VPY, "-c", "import sys,numpy,pandas,scipy,sklearn,torch;print(sys.version.split()[0],numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,torch.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip().split() if r.returncode == 0 else None
v = venv_versions()
if not v or v[1:] != WANT:
    run("pip -q install uv", cwd=W)
    run(f"uv venv --clear --python 3.12 {VENV}", cwd=W)
    run(f"uv pip install --python {VPY} -r {R}/docs/phase3-plan/tools/audit-requirements.txt", cwd=W)
    run(f"uv pip install --python {VPY} -r {R}/configs/phase3/requirements-reference-cpu.lock --extra-index-url https://download.pytorch.org/whl/cpu", cwd=W)
    v = venv_versions()
log("Versions:", v)
assert v and v[1:] == WANT, "Package versions differ from the repository pins"

In [ ]:
# 4. Download the corrected dataset and verify size and SHA-256
D = f"{R}/data/raw/phase3-feasibility-corrected"; os.makedirs(D, exist_ok=True)
Z = f"{D}/CICIDS2017_improved.zip"
if not (os.path.exists(Z) and os.path.getsize(Z) == DATA_BYTES and sha256_file(Z) == DATA_SHA256):
    run(f"curl -L --fail -o {Z} {DATA_URL}", cwd=W)
size, digest = os.path.getsize(Z), sha256_file(Z)
log("size", size, "(expected", DATA_BYTES, ")  sha256", digest)
assert size == DATA_BYTES and digest == DATA_SHA256, "Dataset mismatch: stop. Do not continue with a different file."
log("Dataset verified")

In [ ]:
# 5. Rebuild the derived data (resumable). Order was derived from reading the tools; each step is logged.
T = "docs/phase3-plan/tools"
# Guard: an earlier background run (run_chain.sh) may still be writing the same files in this session.
if subprocess.run(["pgrep", "-f", "[r]un_chain.sh"], capture_output=True).returncode == 0:
    raise RuntimeError("The earlier background chain (run_chain.sh) is still running. Wait for it to finish (tail /kaggle/working/chain.log), then re-run this cell.")
# Adopt steps the earlier background run already completed, so they are not repeated.
old_log = f"{W}/chain.log"
if os.path.exists(old_log):
    cur = None
    for line in open(old_log, errors="ignore"):
        if line.startswith("=== START "): cur = line.split()[2]
        elif line.startswith("=== DONE") and cur and not done(cur): mark(cur); log("adopted completed step from chain.log:", cur)
if not os.path.isdir(f"{W}/docs-phase3-plan-backup"):
    shutil.copytree(f"{R}/docs/phase3-plan", f"{W}/docs-phase3-plan-backup")   # these tools overwrite report files
STEPS = [
    ("A_prepare_inputs",    [VPY, f"{T}/prepare_corrected_component_audit.py"]),
    ("B_components_base",   [VPY, f"{T}/audit_corrected_components.py"]),
    ("C_split_base",        [VPY, f"{T}/audit_candidate_split.py", "--release", "corrected"]),
    ("D_no_headers_74",     [VPY, f"{T}/audit_no_headers_candidate.py"]),
    ("E_split_reuse_proof", [VPY, f"{T}/audit_no_headers_split_reuse.py"]),
    ("F_ownership_74",      [VPY, f"{T}/audit_candidate_ownership.py", "--release", "corrected", "--profile", "no-headers-74",
                             "--benign-policy", "benign-subset-by-component"]),
]
for name, cmd in STEPS:
    if done(name): log("skip (already done):", name); continue
    log("=== START", name)
    run(cmd, logfile=f"{OUT1}/chain-{name}.log")
    mark(name); log("=== DONE", name)
log("All data-rebuild steps finished")

In [ ]:
# 6. Check regenerated files and reports against the committed evidence (does NOT stop; prints a verdict)
import re
ident = json.load(open(f"{R}/docs/phase3-plan/audit-local-data-identities.json"))
def crlf_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 24), b""): h.update(c.replace(bytes([10]), bytes([13, 10])))
    return h.hexdigest()
rows, bad = [], []
for f in ident["files"]:
    p = f"{R}/{f['path']}"
    if not os.path.exists(p): rows.append((f["path"], "not produced (not part of this chain)")); continue
    if sha256_file(p) == f["sha256"]: st = "MATCH"
    elif p.endswith(".csv") and crlf_sha(p) == f["sha256"]: st = "MATCH after Windows line endings (CRLF)"
    else: st = "MISMATCH"; bad.append(f["path"])
    rows.append((f["path"], st))
for path_, st in rows: print(st.ljust(42), path_)
json.dump(rows, open(f"{OUT1}/derived-hash-check.json", "w"), indent=1)

# snapshot the regenerated reports BEFORE restoring the committed (hash-pinned) ones
SNAP = f"{W}/docs-regenerated"
shutil.rmtree(SNAP, ignore_errors=True); shutil.copytree(f"{R}/docs/phase3-plan", SNAP)
changed = subprocess.run("git diff --name-only -- docs/phase3-plan", shell=True, cwd=R, capture_output=True, text=True).stdout.split()
run("git checkout -- docs/phase3-plan", cwd=R)
VOLATILE = re.compile(r"(utc|elapsed|seconds|platform|python|started|completed)", re.I)
def leaves(o, p=""):
    if isinstance(o, dict):
        for k, v in o.items():
            if not VOLATILE.search(k): yield from leaves(v, f"{p}/{k}")
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from leaves(v, f"{p}[{i}]")
    else: yield p, o
report = {}
for rel in changed:
    if not rel.endswith(".json"): continue
    new = dict(leaves(json.load(open(f"{SNAP}/{os.path.basename(rel)}"))))
    old = dict(leaves(json.load(open(f"{R}/{rel}"))))
    diffs = [(k, old.get(k), new.get(k)) for k in sorted(set(old) | set(new)) if old.get(k) != new.get(k)]
    report[rel] = diffs
    print()
    print(f"{os.path.basename(rel)}: {len(set(old)|set(new))} fields compared, {len(diffs)} differ")
    for k, a_, b_ in diffs[:12]:
        print("  ", k)
        print("      committed  =", str(a_)[:90])
        print("      regenerated=", str(b_)[:90])
json.dump({k: [[x, str(a_), str(b_)] for x, a_, b_ in v] for k, v in report.items()}, open(f"{OUT1}/report-content-diff.json", "w"), indent=1)
log("Byte-level mismatches:", len(bad), "-> send this output; the reference run (cell 7) will still refuse until the cause is settled.")

In [ ]:
# 6b. Reproduce Ahmed's exact file bytes (Windows line endings / zip host-OS byte). A file is replaced ONLY if the result
# equals its pinned SHA-256, so no pin is weakened and prepare.py stays untouched.
import zipfile, struct
pins = {f["path"]: f["sha256"] for f in json.load(open(f"{R}/docs/phase3-plan/audit-local-data-identities.json"))["files"]}
def to_crlf(path, out):
    with open(path, "rb") as f, open(out, "wb") as g:
        for c in iter(lambda: f.read(1 << 24), b""): g.write(c.replace(bytes([10]), bytes([13, 10])))
def retag_npz(path, os_byte, ext):
    z = zipfile.ZipFile(path); pos = z.start_dir; n = len(z.infolist()); z.close()
    data = bytearray(open(path, "rb").read())
    for _ in range(n):
        assert data[pos:pos+4] == b"PK"
        data[pos+5] = os_byte
        if ext is not None: data[pos+38:pos+42] = struct.pack("<I", ext)
        nl, el, cl = struct.unpack("<HHH", data[pos+28:pos+34]); pos += 46 + nl + el + cl
    return bytes(data)
res = []
for rel, want in pins.items():
    p = f"{R}/{rel}"
    if not os.path.exists(p) or sha256_file(p) == want: continue
    if rel.endswith(".csv"):
        tmp = p + ".tmp"; to_crlf(p, tmp)
        if sha256_file(tmp) == want: os.replace(tmp, p); res.append((rel, "FIXED: CRLF line endings"))
        else: os.remove(tmp); res.append((rel, "still different"))
    elif rel.endswith(".npz"):
        hit = None
        for os_byte, ext in [(0, None), (0, 0), (0, 0o600 << 16)]:
            b = retag_npz(p, os_byte, ext)
            if hashlib.sha256(b).hexdigest() == want: hit = (os_byte, ext); break
        if hit: open(p + ".tmp", "wb").write(b); os.replace(p + ".tmp", p); res.append((rel, f"FIXED: zip host-OS byte -> {hit[0]}"))
        else: res.append((rel, "still different"))
for rel, st in res: print(st.ljust(40), rel.split("feasibility-")[1])
json.dump(res, open(f"{OUT1}/byte-normalisation.json", "w"), indent=1)

In [ ]:
# A-8. Reference data package (the audited 74-feature package) built with Windows-equivalent zip headers, then every pinned file is verified
P = "data/processed/phase3-reference74-v1"
WRAP = f"{W}/win_zip_wrapper.py"
open(WRAP, "w").write(chr(10).join(["import sys, os, zipfile, runpy", "sys.path.insert(0, os.getcwd())", "_o = zipfile.ZipInfo.__init__", "def _i(self, *a, **k):", "    _o(self, *a, **k); self.create_system = 0", "zipfile.ZipInfo.__init__ = _i", "m = sys.argv[1]; sys.argv = [m] + sys.argv[2:]", "runpy.run_module(m, run_name='__main__', alter_sys=True)", ""]))
if not done("ref_prepare"):
    run([VPY, WRAP, "src.phase3.prepare", "--output", P], logfile=f"{OUT1}/ref-prepare.log"); mark("ref_prepare")
pins = json.load(open(f"{R}/docs/phase3-reference/evidence/preparation-rebuild.json"))["files"]
bad = [n for n, want in pins.items() if not os.path.exists(f"{R}/{P}/{n}") or sha256_file(f"{R}/{P}/{n}") != want]
log("pinned preparation files:", len(pins), "| not matching:", bad)
assert not bad, "The rebuilt package differs from the pinned hashes: stop and send this output."
log("All pinned preparation files match.")

# PART S: adaptive study

In [ ]:
# S-1. Write the study code into the repository (main-study engine unchanged + adaptive extension)
STUDY_SRC = '"""BRMA full study: Benign-Reference Memory Audit for federated class-incremental intrusion detection.\n\nRuns on the audited Phase 3 CPU engine of the project repository (src/phase3, src/phase3_v2; the\n\'core\' study of results/phase3-rigorous-audit-v1, 12 ownership seeds 1-11 and 42).  Engine code is\nimported unchanged; this file adds (i) multi-client label-flip attacks, (ii) end-of-stage memory\ninterventions (BRMA and matched baselines), (iii) a coordinate-wise median aggregation baseline.\nEverything is CPU, single-thread, deterministic; evaluation uses the development bank only.\n\nUsage (from the repository root):  python brma_study.py --out OUT --seeds 1 2 3 4 --workers 4\n"""\nfrom __future__ import annotations\nimport os\nfor _k in (\'OMP_NUM_THREADS\', \'MKL_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'NUMEXPR_NUM_THREADS\'):\n    os.environ[_k] = \'1\'\nimport argparse, copy, json, sys, time, traceback\nimport concurrent.futures as futures\nfrom dataclasses import dataclass, asdict, replace\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nROOT = Path(os.environ.get(\'BRMA_ROOT\', Path.cwd())).resolve()\nsys.path.insert(0, str(ROOT))\nfrom src.phase3.common import rng                      # noqa: E402\nfrom src.phase3.prepare import allocate               # noqa: E402\nfrom src.phase3.model import ReferenceMLP             # noqa: E402\nfrom src.phase3.reference import snapshot, model_identity  # noqa: E402\nfrom src.phase3_v2 import learning as L               # noqa: E402\nfrom src.phase3_v2.data import StageInput             # noqa: E402\nfrom src.phase3_v2.prepare import corrupt             # noqa: E402\n\nBASE = ROOT / \'data/processed/phase3-reference74-v1\'\nSEEDS = tuple(range(1, 12)) + (42,)\nCAPACITY, BENIGN, HULK = 500, 0, 4\n\n# ---------------------------------------------------------------- pre-declared protocol\nARMS = {                     # name: (number of attacking clients k, dose numerator, denominator)\n    \'C\':   (0, 0, 1),        # clean\n    \'K1\':  (1, 1, 2),        # one attacker, 50% of its Hulk rows (identical to the audited study\'s dose 0.5)\n    \'K3\':  (3, 1, 2),        # three coordinated attackers, 50%\n    \'K3H\': (3, 9, 10),       # three coordinated attackers, 90%\n    \'K5\':  (5, 1, 2),        # all five clients, 50%\n}\nBRANCH_METHODS = (\'NONE\', \'BRMA\', \'RAND\', \'LOSS\', \'ORACLE\')   # share stages 1-2, differ by end-of-stage memory action\nFULL_METHODS = (\'FT\', \'EI\', \'DC\', \'DG\', \'MEDIAN\')               # separate full trajectories\nALPHA = 0.05                 # conformal level, fixed by pilots 1-3 on development seeds 101-103\nFIT_CAP, POOL_CAP, MIN_ROWS = 50000, 200000, 20\n\n\n@dataclass(frozen=True)\nclass Settings:              # field names follow src/phase3_v2 so the engine reads them unchanged\n    learner: str = \'DR\'\n    admission: str = \'NONE\'\n    seed: int = 1\n    rounds: int = 12\n    epochs: int = 1\n    learning_rate: float = .001\n    lambda_E: float = 100.\n    lambda_y: float = 1.\n    lambda_z: float = .1\n    q: float = .75\n    eviction_policy: str = \'reservoir\'\n    aggregation: str = \'fedavg\'\n    dose: tuple = (0, 1)\n\n\ndef setup():\n    torch.set_num_threads(1)\n    if torch.get_num_interop_threads() != 1:\n        torch.set_num_interop_threads(1)\n    torch.use_deterministic_algorithms(True)\n    torch.set_float32_matmul_precision(\'highest\')\n\n\n# ---------------------------------------------------------------- engine patches (behaviour-preserving)\ndef insert_refill(client, x, labels, ids, teachers, width):\n    """Reservoir insertion that first fills slots freed by an audit.  Identical to\n    src.phase3.reference.insert_batch whenever no record was removed (size == min(500, seen))."""\n    for i in range(len(labels)):\n        client[\'seen\'] += 1\n        seen = client[\'seen\']\n        if client[\'size\'] < min(CAPACITY, seen - 1):          # a freed slot exists\n            slot = client[\'size\']; client[\'size\'] += 1\n        else:\n            slot = seen - 1 if seen <= CAPACITY else int(client[\'insert\'].integers(0, seen))\n            client[\'size\'] = min(CAPACITY, seen)\n        if slot < CAPACITY:\n            client[\'x\'][slot] = x[i]; client[\'y\'][slot] = labels[i]; client[\'ids\'][slot] = ids[i]\n            client[\'z\'][slot].fill(0); client[\'z\'][slot, :width] = teachers[i]; client[\'width\'][slot] = width\n\n\nL.insert_batch = insert_refill\n_FEDAVG = L.aggregate\n\n\ndef median_aggregate(broadcast, updates, counts):\n    """Coordinate-wise median of the active client models (robust-aggregation baseline)."""\n    active = [u for u, n in zip(updates, counts) if n]\n    if len(active) < 3:\n        return _FEDAVG(broadcast, updates, counts)\n    combined = {k: torch.quantile(torch.stack([u[k] for u in active]), .5, dim=0) for k in broadcast}\n    return combined, [1. / len(active) if n else 0. for n in counts]\n\n\n# ---------------------------------------------------------------- data and attacks\ndef corrupt_multi(clean, ids, owner, seed, stage, k, num, den):\n    """k-client nested label flip of DoS Hulk to benign in stage 2.  The first attacker and its row order\n    use the audited single-attacker streams (rng 2 and rng 3), so k = 1 reproduces src.phase3_v2.prepare.corrupt."""\n    supplied = clean.copy()\n    info = dict(stage=stage, k=k, dose=[num, den], attackers=[], poison_ids=[])\n    if stage != 2 or k == 0 or num == 0:\n        return supplied, info\n    if k == 1 and (num, den) in {(1, 10), (3, 10), (1, 2)}:\n        supplied, a = corrupt(clean, ids, owner, seed, stage, (num, den))\n        info.update(attackers=[a[\'attacked_client\']], poison_ids=a[\'poison_ids\'])\n        return supplied, info\n    counts = np.bincount(owner[clean == HULK], minlength=5)\n    eligible = np.flatnonzero(counts >= 10)\n    first = int(rng(2, seed).choice(eligible))\n    rest = [int(c) for c in rng(12, seed).permutation(eligible) if c != first]\n    attackers = [first] + rest[:k - 1]\n    poison = []\n    for a in attackers:\n        source = np.flatnonzero((owner == a) & (clean == HULK))\n        source = source[np.argsort(ids[source], kind=\'stable\')]\n        prefix = rng(3, seed, a).permutation(source)\n        chosen = prefix[:num * len(source) // den]\n        supplied[chosen] = BENIGN; poison.extend(str(int(v)) for v in ids[chosen])\n    info.update(attackers=attackers, poison_ids=poison)\n    return supplied, info\n\n\ndef inputs(seed, arm, out):\n    with np.load(out / \'ownership-source.npz\', allow_pickle=False) as f:\n        source_ids, components, labels = f[\'ids\'], f[\'components\'], f[\'labels\']\n    owners, _ = allocate(components, labels, seed)\n    order = np.argsort(source_ids, kind=\'stable\')\n    k, num, den = ARMS[arm]\n    data, attack = [], None\n    for stage in range(1, 5):\n        folder = BASE / f\'train/E{stage}\'\n        x = np.load(folder / \'x.npy\', mmap_mode=\'r\', allow_pickle=False)\n        ids = np.load(folder / \'ids.npy\', allow_pickle=False)\n        clean = np.load(folder / \'y.npy\', allow_pickle=False)\n        index = order[np.searchsorted(source_ids[order], ids)]\n        assert np.array_equal(source_ids[index], ids) and np.array_equal(labels[index], clean)\n        owner = owners[index].astype(\'i1\')\n        y, info = corrupt_multi(clean, ids, owner, seed, stage, k, num, den)\n        if stage == 2:\n            attack = info\n        data.append(StageInput(stage, x, y, ids, owner))\n    return data, attack\n\n\ndef evaluate(model, slots, stage):\n    """Frozen development bank only (as in the audited study)."""\n    folder = BASE / \'development\'\n    x = np.load(folder / \'x.npy\', mmap_mode=\'r\', allow_pickle=False)\n    y = np.load(folder / \'y.npy\', mmap_mode=\'r\', allow_pickle=False)\n    keep = np.flatnonzero(np.isin(y, slots))\n    matrix = np.zeros((16, 16), dtype=np.int64)\n    model.eval()\n    with torch.no_grad():\n        for start in range(0, len(keep), 4096):\n            ix = keep[start:start + 4096]\n            pred = np.asarray(slots)[model(torch.from_numpy(x[ix].copy())).argmax(1).numpy()]\n            np.add.at(matrix, (y[ix], pred), 1)\n    return dict(stage=stage, slots=list(slots), confusion=matrix.tolist(), model_sha256=model_identity(model, slots))\n\n\n# ---------------------------------------------------------------- BRMA: server statistics and audit\ndef ledoit_wolf_precision(X):\n    from sklearn.covariance import LedoitWolf\n    return LedoitWolf(assume_centered=True).fit(X).precision_\n\n\ndef maha_sq(Z, mu, P, chunk=65536):\n    out = np.empty(len(Z))\n    for i in range(0, len(Z), chunk):\n        D = Z[i:i + chunk] - mu; out[i:i + chunk] = np.einsum(\'ij,ij->i\', D @ P, D)\n    return out\n\n\nclass Auditor:\n    """Server-side Gaussian statistics.  Benign: stage-1 rows only (before any attack), split into a\n    reference half (statistics) and a calibration half (conformal threshold).  Attack classes: rows the\n    clients label as that class in stages <= t (supplied labels; flipped rows are labelled benign and\n    therefore never enter attack statistics).  Score = 0.5 (d_benign^2 - min_c d_c^2) under one shared\n    Ledoit-Wolf covariance (tied-covariance likelihood ratio)."""\n\n    def __init__(self, data, seed):\n        self.data, self.seed = data, seed\n        x1 = np.asarray(data[0].x)\n        self.mu0, self.sd0 = x1.mean(0), x1.std(0) + 1e-6\n        ben = rng(41, seed).permutation(np.flatnonzero(data[0].y == BENIGN))\n        self.Zref = self.std(x1[ben[: len(ben) // 2]]); self.Zcal = self.std(x1[ben[len(ben) // 2:]])\n        self.models = {}\n\n    def std(self, X):\n        return (np.asarray(X, dtype=np.float64) - self.mu0) / self.sd0\n\n    def model(self, t):\n        if t in self.models:\n            return self.models[t]\n        means, centred = {}, [self.Zref - self.Zref.mean(0)]\n        for c in range(1, 16):\n            parts = [np.asarray(self.data[s].x)[self.data[s].y == c] for s in range(t)]\n            X = np.concatenate(parts) if parts else np.zeros((0, 74))\n            if len(X) < MIN_ROWS:\n                continue\n            if len(X) > FIT_CAP:\n                X = X[rng(42, self.seed, t, c).choice(len(X), FIT_CAP, replace=False)]\n            Z = self.std(X); means[c] = Z.mean(0); centred.append(Z - means[c])\n        pooled = np.concatenate(centred)\n        if len(pooled) > POOL_CAP:\n            pooled = pooled[rng(43, self.seed, t).choice(len(pooled), POOL_CAP, replace=False)]\n        P = ledoit_wolf_precision(pooled)\n        m = dict(P=P, mub=self.Zref.mean(0), means=means)\n        m[\'tau\'] = conformal(self.score_z(self.Zcal, m), ALPHA)\n        self.models[t] = m\n        return m\n\n    def score_z(self, Z, m):\n        if not len(Z):\n            return np.zeros(0)\n        da = np.min([maha_sq(Z, mu, m[\'P\']) for mu in m[\'means\'].values()], axis=0)\n        return 0.5 * (maha_sq(Z, m[\'mub\'], m[\'P\']) - da)\n\n    def flags(self, X, t):\n        m = self.model(t); s = self.score_z(self.std(X), m)\n        return s > m[\'tau\'], s, m[\'tau\']\n\n\ndef conformal(cal, alpha):\n    n = len(cal); return float(np.quantile(cal, min(1., np.ceil((n + 1) * (1 - alpha)) / n), method=\'higher\'))\n\n\ndef remove_rows(memory, drop):\n    keep = np.setdiff1d(np.arange(memory[\'size\']), drop)\n    for key in (\'x\', \'y\', \'ids\', \'z\', \'width\'):\n        memory[key][:len(keep)] = memory[key][keep]\n    memory[\'size\'] = len(keep)\n\n\ndef audit(method, model, clients, slots, t, auditor, poison_ids, seed):\n    """End-of-stage memory action.  RAND and LOSS remove exactly as many benign-labelled records per\n    client as BRMA flags on the same memory (matched removal budget)."""\n    rows = []\n    benign_slot = slots.index(BENIGN)\n    for k, c in enumerate(clients):\n        m = c.replay\n        if m is None or m[\'size\'] == 0:\n            rows.append(dict(client=k, size=0)); continue\n        n = m[\'size\']; y, ids = m[\'y\'][:n], m[\'ids\'][:n]\n        is_poison = np.isin(ids, poison_ids); ben = np.flatnonzero(y == BENIGN)\n        flag, _, tau = auditor.flags(m[\'x\'][:n][ben], t) if len(ben) else (np.zeros(0, bool), None, None)\n        budget = int(flag.sum())\n        if method == \'BRMA\':\n            drop = ben[flag]\n        elif method == \'RAND\':\n            drop = rng(44, seed, t, k).choice(ben, budget, replace=False) if budget else np.zeros(0, int)\n        elif method == \'LOSS\':\n            model.eval()\n            with torch.no_grad():\n                loss = F.cross_entropy(model(torch.from_numpy(m[\'x\'][:n][ben].copy())),\n                                       torch.full((len(ben),), benign_slot, dtype=torch.int64), reduction=\'none\').numpy()\n            drop = ben[np.lexsort((ids[ben], -loss))[:budget]]\n        elif method == \'ORACLE\':\n            drop = np.flatnonzero(is_poison)\n        else:\n            drop = np.zeros(0, int)\n        drop = np.asarray(drop, dtype=np.int64)\n        rows.append(dict(client=k, size=int(n), benign_labelled=int(len(ben)), poison_before=int(is_poison.sum()),\n                         brma_flagged=budget, brma_flagged_poison=int(is_poison[ben][flag].sum()) if len(ben) else 0,\n                         removed=int(len(drop)), removed_poison=int(is_poison[drop].sum()),\n                         removed_genuine=int(len(drop) - is_poison[drop].sum()), tau=tau))\n        remove_rows(m, drop)\n    return rows\n\n\n# ---------------------------------------------------------------- training\ndef run_stages(model, clients, slots, data, s, start, end, folder_tag=None):\n    stages = []\n    L.aggregate = median_aggregate if s.aggregation == \'median\' else _FEDAVG\n    for stage in range(start, end + 1):\n        d = data[stage - 1]\n        new_codes = [int(c) for c in np.unique(d.y) if int(c) not in slots]\n        slots.extend(new_codes)\n        if model is None:\n            model = ReferenceMLP(len(slots), s.seed)\n        elif new_codes:\n            model.expand(len(slots), stage, s.seed)\n        if s.learner == \'EI\':\n            for c in clients:\n                L.expand_importance(c, model)\n        L.admit(model, d, slots, set(new_codes), clients, s, [])\n        for r in range(1, s.rounds + 1):\n            broadcast = snapshot(model); updates = []\n            for i, c in enumerate(clients):\n                update, _ = L.train_client(broadcast, model, c, d, slots, s, r, i); updates.append(update)\n            combined, _ = L.aggregate(broadcast, updates, [len(c.accepted) for c in clients])\n            model.load_state_dict(combined)\n        if s.learner == \'EI\':\n            L.commit_importance(model, clients, d, slots, s, [])\n        stages.append(evaluate(model, slots, stage))\n    return model, stages\n\n\ndef memory_census(clients, poison_ids):\n    out = []\n    for c in clients:\n        m = c.replay\n        if m is None:\n            out.append(None); continue\n        n = m[\'size\']; p = np.isin(m[\'ids\'][:n], poison_ids)\n        out.append(dict(size=int(n), poison=int(p.sum()), benign_labelled=int((m[\'y\'][:n] == BENIGN).sum())))\n    return out\n\n\ndef save(out, key, payload, model, slots):\n    folder = out / \'runs\' / key; folder.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(folder / \'E4-weights.npz\', **{k: v.numpy() for k, v in snapshot(model).items()}, source_codes=np.asarray(slots))\n    tmp = folder / \'result.json.tmp\'; tmp.write_text(json.dumps(payload)); tmp.replace(folder / \'result.json\')\n\n\ndef job_key(seed, arm, method):\n    return f\'s{seed}-{arm}-{method}\'\n\n\ndef done(out, seed, arm, method):\n    return (out / \'runs\' / job_key(seed, arm, method) / \'result.json\').exists()\n\n\ndef worker(job):\n    setup()\n    kind, seed, arm, out = job[\'kind\'], job[\'seed\'], job[\'arm\'], Path(job[\'out\'])\n    t0 = time.perf_counter()\n    data, attack = inputs(seed, arm, out)\n    poison_ids = np.asarray([int(v) for v in attack[\'poison_ids\']], dtype=np.uint64)\n    base = dict(seed=seed, arm=arm, arm_spec=ARMS[arm], attack=dict(attackers=attack[\'attackers\'], k=attack[\'k\'],\n                dose=attack[\'dose\'], n_poison=len(poison_ids)), alpha=ALPHA, development_bank_only=True)\n    if kind == \'full\':\n        method = job[\'method\']\n        s = Settings(seed=seed, dose=ARMS[arm][1:],\n                     learner=method if method in (\'FT\', \'EI\') else \'DR\',\n                     admission=method if method in (\'DC\', \'DG\') else \'NONE\',\n                     aggregation=\'median\' if method == \'MEDIAN\' else \'fedavg\')\n        clients = [L.client_state(s, i) for i in range(5)]\n        model, stages = run_stages(None, clients, [], data, s, 1, 4)\n        slots = stages[-1][\'slots\']\n        save(out, job_key(seed, arm, method), dict(base, method=method, settings=asdict(s), stages=stages,\n             memory=memory_census(clients, poison_ids), audits=[], elapsed_seconds=time.perf_counter() - t0), model, slots)\n        return [job_key(seed, arm, method)]\n    # branch tree: shared stages 1-2, then one suffix per memory action\n    s = Settings(seed=seed, dose=ARMS[arm][1:])\n    clients = [L.client_state(s, i) for i in range(5)]\n    slots = []\n    model, prefix = run_stages(None, clients, slots, data, s, 1, 2)\n    census2 = memory_census(clients, poison_ids)\n    prefix_seconds = time.perf_counter() - t0\n    auditor = Auditor(data, seed)\n    keys = []\n    for method in BRANCH_METHODS:\n        if done(out, seed, arm, method):\n            continue\n        t1 = time.perf_counter()\n        bm, bc, bs = copy.deepcopy(model), copy.deepcopy(clients), list(slots)\n        audits = []\n        audits.append(dict(after_stage=2, clients=audit(method, bm, bc, bs, 2, auditor, poison_ids, seed)))\n        bm, st3 = run_stages(bm, bc, bs, data, s, 3, 3)\n        audits.append(dict(after_stage=3, clients=audit(method, bm, bc, bs, 3, auditor, poison_ids, seed)))\n        bm, st4 = run_stages(bm, bc, bs, data, s, 4, 4)\n        save(out, job_key(seed, arm, method), dict(base, method=method, settings=asdict(s), stages=prefix + st3 + st4,\n             memory_after_stage2=census2, memory=memory_census(bc, poison_ids), audits=audits,\n             elapsed_seconds=prefix_seconds + time.perf_counter() - t1, shared_prefix_seconds=prefix_seconds), bm, bs)\n        keys.append(job_key(seed, arm, method))\n    return keys\n\n\ndef build_jobs(out, seeds):\n    jobs = []\n    for seed in seeds:\n        for arm in ARMS:\n            if not all(done(out, seed, arm, m) for m in BRANCH_METHODS):\n                jobs.append(dict(kind=\'tree\', seed=seed, arm=arm, out=str(out)))\n            for m in FULL_METHODS:\n                if not done(out, seed, arm, m):\n                    jobs.append(dict(kind=\'full\', seed=seed, arm=arm, method=m, out=str(out)))\n    # clean replay trees first (validation gate), then by seed\n    jobs.sort(key=lambda j: (j[\'arm\'] != \'C\' or j[\'kind\'] != \'tree\', j[\'seed\'], j[\'kind\'] != \'tree\'))\n    return jobs\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--out\', type=Path, required=True); ap.add_argument(\'--seeds\', type=int, nargs=\'+\', default=list(SEEDS))\n    ap.add_argument(\'--workers\', type=int, default=4); ap.add_argument(\'--hours\', type=float, default=11.0)\n    a = ap.parse_args(); out = a.out.resolve(); out.mkdir(parents=True, exist_ok=True)\n    if not (out / \'ownership-source.npz\').exists():\n        with np.load(BASE / \'builder/ownership.npz\', allow_pickle=False) as f:\n            positions, components = f[\'source_positions\'], f[\'canonical_components\']\n        with np.load(ROOT / \'data/raw/phase3-feasibility-corrected/no-headers-74/conditional-components-120.npz\', allow_pickle=False) as f:\n            ids, labels = f[\'row_ids\'][positions], f[\'labels\'][positions].astype(np.int64)\n        np.savez_compressed(out / \'ownership-source.npz\', ids=ids, components=components, labels=labels)\n    (out / \'protocol.json\').write_text(json.dumps(dict(arms=ARMS, branch_methods=BRANCH_METHODS, full_methods=FULL_METHODS,\n        alpha=ALPHA, seeds=a.seeds, engine=\'src/phase3 + src/phase3_v2 (audited core), 12 rounds x 1 epoch, lr .001, lambda_E 100, lambda_y 1, lambda_z .1\'), indent=1))\n    jobs = build_jobs(out, a.seeds); t0 = time.time(); finished = 0\n    print(f\'{len(jobs)} jobs pending for seeds {a.seeds}\', flush=True)\n    with futures.ProcessPoolExecutor(max_workers=a.workers) as pool:\n        pending = {pool.submit(worker, j): j for j in jobs}\n        for f in futures.as_completed(pending):\n            j = pending[f]\n            try:\n                keys = f.result()\n            except Exception:\n                (out / f"failure-{j[\'kind\']}-s{j[\'seed\']}-{j[\'arm\']}-{j.get(\'method\', \'tree\')}.txt").write_text(traceback.format_exc())\n                print(\'FAILED\', j, flush=True); continue\n            finished += 1; el = time.time() - t0\n            print(f"[{finished}/{len(jobs)}] {j[\'kind\']} s{j[\'seed\']} {j[\'arm\']} {j.get(\'method\', \'\')} -> {len(keys)} results | "\n                  f"{el / 60:.1f} min elapsed, ~{el / finished * (len(jobs) - finished) / 60:.0f} min left", flush=True)\n    print(\'complete\' if all(done(out, s, ar, m) for s in a.seeds for ar in ARMS for m in BRANCH_METHODS + FULL_METHODS) else \'incomplete: re-run to resume\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
ADAPT_SRC = '"""BRMA adaptive-attack study (extension of brma_study.py; same audited engine, same 12 seeds, CIC-IDS2017).\n\nAdds, without changing brma_study.py:\n  * two adaptive attacks by attackers who know BRMA:\n      sel - each attacker scores its own DoS Hulk records with its own copy of the BRMA likelihood ratio\n            (tied Gaussian, Ledoit-Wolf, fitted on its own genuine benign and Hulk records) and flips the\n            floor(rho * n) records that look MOST benign;\n      mix - the usual random nested flip, plus feature camouflage: each flipped record is moved a fraction\n            GAMMA of the way towards the attacker\'s own benign mean (x\' = x + GAMMA (m_benign - x));\n  * a self-sufficient baseline LOSS5 that removes a FIXED 5% of benign-labelled memory with the highest\n    loss under the current model (no BRMA information).\nPre-declared tests (Holm over four, one-sided paired Wilcoxon over 12 seeds):\n  T7 BRMA < NONE on H at S3H;  T8 BRMA < NONE on H at M3H;  T9 BRMA < LOSS5 on H at K3H;  T10 BRMA < LOSS5 on H at S3H.\n\nUsage (from the repository root, next to brma_study.py):  python brma_adaptive.py --out OUT --workers 4\n"""\nfrom __future__ import annotations\nimport os\nfor _k in (\'OMP_NUM_THREADS\', \'MKL_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'NUMEXPR_NUM_THREADS\'):\n    os.environ[_k] = \'1\'\nimport argparse, copy, json, sys, time, traceback\nimport concurrent.futures as futures\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport brma_study as B                                   # noqa: E402  (frozen main-study code, imported unchanged)\nfrom src.phase3.common import rng                       # noqa: E402\n\nARMS = {                    # name: (attackers k, dose numerator, denominator, attack mode)\n    \'C\':   (0, 0, 1, \'std\'),\n    \'K3H\': (3, 9, 10, \'std\'),\n    \'K5\':  (5, 1, 2, \'std\'),\n    \'S3H\': (3, 9, 10, \'sel\'),\n    \'S5\':  (5, 1, 2, \'sel\'),\n    \'M3H\': (3, 9, 10, \'mix\'),\n    \'M5\':  (5, 1, 2, \'mix\'),\n}\nBRANCH_METHODS = (\'NONE\', \'BRMA\', \'RAND\', \'LOSS\', \'ORACLE\', \'LOSS5\')\nFULL_METHODS = (\'DC\', \'MEDIAN\')\nGAMMA, FIXED_FRACTION, HULK, BENIGN = 0.5, 0.05, B.HULK, B.BENIGN\n\n\n# ---------------------------------------------------------------- attacker-side BRMA copy\ndef attacker_score(Xb, Xh, X, seed, a):\n    """The attacker\'s own likelihood ratio log p(x|Hulk) - log p(x|benign) with a tied Ledoit-Wolf covariance."""\n    from sklearn.covariance import LedoitWolf\n    cap = 50000\n    if len(Xb) > cap:\n        Xb = Xb[rng(45, seed, a).choice(len(Xb), cap, replace=False)]\n    m, s = Xb.mean(0), Xb.std(0) + 1e-6\n    Zb, Zh, Z = (np.asarray(Xb, np.float64) - m) / s, (np.asarray(Xh, np.float64) - m) / s, (np.asarray(X, np.float64) - m) / s\n    mb, mh = Zb.mean(0), Zh.mean(0)\n    P = LedoitWolf(assume_centered=True).fit(np.concatenate([Zb - mb, Zh - mh])).precision_\n    d = lambda Q, mu: np.einsum(\'ij,ij->i\', (Q - mu) @ P, Q - mu)\n    return 0.5 * (d(Z, mb) - d(Z, mh))                  # high = looks like Hulk, low = looks benign\n\n\ndef corrupt_adaptive(clean, x, ids, owner, seed, stage, k, num, den, mode, benign_pool):\n    """Returns supplied labels, possibly modified features (stage 2 copy) and attack info.\n    benign_pool[a] = the attacker\'s genuine benign records (stages 1-2) in raw feature space."""\n    if mode == \'std\' or stage != 2 or k == 0 or num == 0:\n        y, info = B.corrupt_multi(clean, ids, owner, seed, stage, k, num, den)\n        info[\'mode\'] = mode\n        return y, x, info\n    supplied = clean.copy(); xs = np.array(x, dtype=np.float32, copy=True)\n    counts = np.bincount(owner[clean == HULK], minlength=5)\n    eligible = np.flatnonzero(counts >= 10)\n    first = int(rng(2, seed).choice(eligible))\n    rest = [int(c) for c in rng(12, seed).permutation(eligible) if c != first]\n    attackers = [first] + rest[:k - 1]\n    poison, moved = [], 0\n    for a in attackers:\n        source = np.flatnonzero((owner == a) & (clean == HULK))\n        source = source[np.argsort(ids[source], kind=\'stable\')]\n        q = num * len(source) // den\n        if mode == \'sel\':\n            sc = attacker_score(benign_pool[a], xs[source], xs[source], seed, a)\n            chosen = source[np.lexsort((ids[source], sc))[:q]]          # most benign-looking first\n        else:                                                           # \'mix\': same nested random order as std\n            chosen = rng(3, seed, a).permutation(source)[:q]\n            mb = np.asarray(benign_pool[a], np.float64).mean(0)\n            xs[chosen] = (xs[chosen] + GAMMA * (mb - xs[chosen])).astype(np.float32); moved += len(chosen)\n        supplied[chosen] = BENIGN; poison.extend(str(int(v)) for v in ids[chosen])\n    return supplied, xs, dict(stage=stage, k=k, dose=[num, den], attackers=attackers, poison_ids=poison, mode=mode, moved=moved)\n\n\ndef inputs(seed, arm, out):\n    """As brma_study.inputs, with adaptive attacks."""\n    with np.load(Path(out) / \'ownership-source.npz\', allow_pickle=False) as f:\n        source_ids, components, labels = f[\'ids\'], f[\'components\'], f[\'labels\']\n    owners, _ = B.allocate(components, labels, seed)\n    order = np.argsort(source_ids, kind=\'stable\')\n    k, num, den, mode = ARMS[arm]\n    raw = []\n    for stage in range(1, 5):\n        folder = B.BASE / f\'train/E{stage}\'\n        x = np.load(folder / \'x.npy\', mmap_mode=\'r\', allow_pickle=False)\n        ids = np.load(folder / \'ids.npy\', allow_pickle=False)\n        clean = np.load(folder / \'y.npy\', allow_pickle=False)\n        index = order[np.searchsorted(source_ids[order], ids)]\n        assert np.array_equal(source_ids[index], ids) and np.array_equal(labels[index], clean)\n        raw.append((stage, x, ids, clean, owners[index].astype(\'i1\')))\n    benign_pool = {}\n    if mode != \'std\':\n        for a in range(5):\n            parts = [np.asarray(x)[(o == a) & (y == BENIGN)] for (s, x, ids, y, o) in raw[:2]]\n            pool = np.concatenate(parts)\n            if len(pool) < 20:                                          # attacker without benign traffic: use all its stage-2 non-Hulk rows\n                s, x, ids, y, o = raw[1]; pool = np.asarray(x)[(o == a) & (y != HULK)]\n            benign_pool[a] = pool\n    data, attack = [], None\n    for stage, x, ids, clean, owner in raw:\n        y, xs, info = corrupt_adaptive(clean, x, ids, owner, seed, stage, k, num, den, mode, benign_pool)\n        if stage == 2:\n            attack = info\n        data.append(B.StageInput(stage, xs, y, ids, owner))\n    return data, attack\n\n\n# ---------------------------------------------------------------- memory actions (adds LOSS5)\ndef audit(method, model, clients, slots, t, auditor, poison_ids, seed):\n    if method != \'LOSS5\':\n        return B.audit(method, model, clients, slots, t, auditor, poison_ids, seed)\n    rows = []; bslot = slots.index(BENIGN)\n    for k, c in enumerate(clients):\n        m = c.replay\n        if m is None or m[\'size\'] == 0:\n            rows.append(dict(client=k, size=0)); continue\n        n = m[\'size\']; y, ids = m[\'y\'][:n], m[\'ids\'][:n]\n        is_p = np.isin(ids, poison_ids); ben = np.flatnonzero(y == BENIGN)\n        budget = int(np.floor(FIXED_FRACTION * len(ben) + 0.5))\n        drop = np.zeros(0, np.int64)\n        if budget:\n            model.eval()\n            with torch.no_grad():\n                loss = F.cross_entropy(model(torch.from_numpy(m[\'x\'][:n][ben].copy())), torch.full((len(ben),), bslot, dtype=torch.int64),\n                                       reduction=\'none\').numpy()\n            drop = ben[np.lexsort((ids[ben], -loss))[:budget]].astype(np.int64)\n        rows.append(dict(client=k, size=int(n), benign_labelled=int(len(ben)), poison_before=int(is_p.sum()), budget=budget,\n                         removed=int(len(drop)), removed_poison=int(is_p[drop].sum()), removed_genuine=int(len(drop) - is_p[drop].sum())))\n        B.remove_rows(m, drop)\n    return rows\n\n\ndef brma_detection(clients, t, auditor, poison_ids):\n    """BRMA detection on the memory as it stands (recorded for every branch, no removal)."""\n    out = []\n    for c in clients:\n        m = c.replay\n        if m is None or m[\'size\'] == 0:\n            continue\n        n = m[\'size\']; ben = np.flatnonzero(m[\'y\'][:n] == BENIGN)\n        if not len(ben):\n            continue\n        flag, _, _ = auditor.flags(m[\'x\'][:n][ben], t); is_p = np.isin(m[\'ids\'][:n][ben], poison_ids)\n        out.append(dict(benign_labelled=int(len(ben)), poison=int(is_p.sum()), flagged=int(flag.sum()), flagged_poison=int((flag & is_p).sum())))\n    return out\n\n\n# ---------------------------------------------------------------- jobs\ndef worker(job):\n    B.setup()\n    kind, seed, arm, out = job[\'kind\'], job[\'seed\'], job[\'arm\'], Path(job[\'out\'])\n    t0 = time.perf_counter()\n    data, attack = inputs(seed, arm, out)\n    poison_ids = np.asarray([int(v) for v in attack[\'poison_ids\']], dtype=np.uint64)\n    base = dict(seed=seed, arm=arm, arm_spec=ARMS[arm], attack=dict(attackers=attack[\'attackers\'], k=attack[\'k\'], dose=attack[\'dose\'], mode=attack.get(\'mode\'),\n                n_poison=len(poison_ids), moved=attack.get(\'moved\', 0)), alpha=B.ALPHA, gamma=GAMMA, fixed_fraction=FIXED_FRACTION, development_bank_only=True)\n    if kind == \'full\':\n        method = job[\'method\']\n        s = B.Settings(seed=seed, dose=ARMS[arm][1:3], admission=method if method in (\'DC\', \'DG\') else \'NONE\',\n                       aggregation=\'median\' if method == \'MEDIAN\' else \'fedavg\')\n        clients = [B.L.client_state(s, i) for i in range(5)]\n        model, stages = B.run_stages(None, clients, [], data, s, 1, 4)\n        B.save(out, B.job_key(seed, arm, method), dict(base, method=method, settings=asdict(s), stages=stages,\n               memory=B.memory_census(clients, poison_ids), audits=[], elapsed_seconds=time.perf_counter() - t0), model, stages[-1][\'slots\'])\n        return [B.job_key(seed, arm, method)]\n    s = B.Settings(seed=seed, dose=ARMS[arm][1:3])\n    clients = [B.L.client_state(s, i) for i in range(5)]; slots = []\n    model, prefix = B.run_stages(None, clients, slots, data, s, 1, 2)\n    census2 = B.memory_census(clients, poison_ids); prefix_seconds = time.perf_counter() - t0\n    auditor = B.Auditor(data, seed); keys = []\n    for method in BRANCH_METHODS:\n        if B.done(out, seed, arm, method):\n            continue\n        t1 = time.perf_counter()\n        bm, bc, bs = copy.deepcopy(model), copy.deepcopy(clients), list(slots)\n        det2 = brma_detection(bc, 2, auditor, poison_ids)\n        audits = [dict(after_stage=2, clients=audit(method, bm, bc, bs, 2, auditor, poison_ids, seed), brma_detection=det2)]\n        bm, st3 = B.run_stages(bm, bc, bs, data, s, 3, 3)\n        det3 = brma_detection(bc, 3, auditor, poison_ids)\n        audits.append(dict(after_stage=3, clients=audit(method, bm, bc, bs, 3, auditor, poison_ids, seed), brma_detection=det3))\n        bm, st4 = B.run_stages(bm, bc, bs, data, s, 4, 4)\n        B.save(out, B.job_key(seed, arm, method), dict(base, method=method, settings=asdict(s), stages=prefix + st3 + st4,\n               memory_after_stage2=census2, memory=B.memory_census(bc, poison_ids), audits=audits,\n               elapsed_seconds=prefix_seconds + time.perf_counter() - t1, shared_prefix_seconds=prefix_seconds), bm, bs)\n        keys.append(B.job_key(seed, arm, method))\n    return keys\n\n\ndef build_jobs(out, seeds):\n    jobs = []\n    for seed in seeds:\n        for arm in ARMS:\n            if not all(B.done(out, seed, arm, m) for m in BRANCH_METHODS):\n                jobs.append(dict(kind=\'tree\', seed=seed, arm=arm, out=str(out)))\n            for m in FULL_METHODS:\n                if not B.done(out, seed, arm, m):\n                    jobs.append(dict(kind=\'full\', seed=seed, arm=arm, method=m, out=str(out)))\n    jobs.sort(key=lambda j: (seeds.index(j[\'seed\']), j[\'kind\'] != \'tree\'))\n    return jobs\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--out\', type=Path, required=True); ap.add_argument(\'--seeds\', type=int, nargs=\'+\', default=list(B.SEEDS))\n    ap.add_argument(\'--workers\', type=int, default=4)\n    a = ap.parse_args(); out = a.out.resolve(); out.mkdir(parents=True, exist_ok=True)\n    if not (out / \'ownership-source.npz\').exists():\n        with np.load(B.BASE / \'builder/ownership.npz\', allow_pickle=False) as f:\n            positions, components = f[\'source_positions\'], f[\'canonical_components\']\n        with np.load(B.ROOT / \'data/raw/phase3-feasibility-corrected/no-headers-74/conditional-components-120.npz\', allow_pickle=False) as f:\n            ids, labels = f[\'row_ids\'][positions], f[\'labels\'][positions].astype(np.int64)\n        np.savez_compressed(out / \'ownership-source.npz\', ids=ids, components=components, labels=labels)\n    (out / \'protocol.json\').write_text(json.dumps(dict(arms=ARMS, branch_methods=BRANCH_METHODS, full_methods=FULL_METHODS, alpha=B.ALPHA,\n        gamma=GAMMA, fixed_fraction=FIXED_FRACTION, seeds=a.seeds, tests=[\'T7 BRMA<NONE H S3H\', \'T8 BRMA<NONE H M3H\', \'T9 BRMA<LOSS5 H K3H\',\n        \'T10 BRMA<LOSS5 H S3H\'], engine=\'brma_study.py engine unchanged\'), indent=1))\n    jobs = build_jobs(out, a.seeds); t0 = time.time(); fin = 0\n    print(f\'{len(jobs)} jobs pending for seeds {a.seeds}\', flush=True)\n    with futures.ProcessPoolExecutor(max_workers=a.workers) as pool:\n        pend = {pool.submit(worker, j): j for j in jobs}\n        for f in futures.as_completed(pend):\n            j = pend[f]\n            try:\n                keys = f.result()\n            except Exception:\n                (out / f"failure-{j[\'kind\']}-s{j[\'seed\']}-{j[\'arm\']}-{j.get(\'method\', \'tree\')}.txt").write_text(traceback.format_exc())\n                print(\'FAILED\', j, flush=True); continue\n            fin += 1; el = time.time() - t0\n            print(f"[{fin}/{len(jobs)}] {j[\'kind\']} s{j[\'seed\']} {j[\'arm\']} {j.get(\'method\', \'\')} -> {len(keys)} results | "\n                  f"{el / 60:.1f} min elapsed, ~{el / fin * (len(jobs) - fin) / 60:.0f} min left", flush=True)\n    print(\'complete\' if not build_jobs(out, a.seeds) else \'incomplete: re-run to resume\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
open(f"{R}/brma_study.py", "w").write(STUDY_SRC); open(f"{R}/brma_adaptive.py", "w").write(ADAPT_SRC)
log("brma_study.py sha256", hashlib.sha256(STUDY_SRC.encode()).hexdigest()[:16], "| brma_adaptive.py sha256", hashlib.sha256(ADAPT_SRC.encode()).hexdigest()[:16])
SOUT = f"{W}/brma-adaptive"; os.makedirs(SOUT, exist_ok=True)

In [ ]:
# S-2. Run all 12 seeds (resumable; prints progress and time left)
run([VPY, "brma_adaptive.py", "--out", SOUT, "--workers", str(os.cpu_count())], cwd=R, logfile=f"{SOUT}/run.log", check=False)

In [ ]:
# S-3. Summary: H, macro-F1, detection, pre-declared tests
import numpy as np, pandas as pd, glob
from scipy.stats import wilcoxon
sys.path.insert(0, f"{R}/scripts"); from phase3_evidence import confusion_metrics
tax = json.load(open(f"{R}/configs/phase3/taxonomy-v1.json"))
ELIG = [c["source_code"] for c in tax["classes"] if c["scoring_eligible"]]; ARR = {c["source_code"]: c["arrival_stage"] for c in tax["classes"]}
rows = []
for p in glob.glob(f"{SOUT}/runs/*/result.json"):
    r = json.load(open(p))
    st = [confusion_metrics(np.asarray(s["confusion"]), [c for c in ELIG if ARR[c] <= s["stage"]]) for s in r["stages"]]
    det = [d for a in r.get("audits", []) if a["after_stage"] == 2 for d in a.get("brma_detection", [])]
    rows.append(dict(seed=r["seed"], arm=r["arm"], method=r["method"], macro_f1=st[-1]["macro_f1"], accuracy=st[-1]["accuracy"],
                     e3_asr=st[2]["hulk_asr"], e4_asr=st[3]["hulk_asr"], det_poison=sum(d["poison"] for d in det), det_flag=sum(d["flagged_poison"] for d in det)))
df = pd.DataFrame(rows)
cl = df[df.arm == "C"].set_index(["seed", "method"])
df["H"] = [.5 * ((r.e3_asr - cl.loc[(r.seed, r.method), "e3_asr"]) + (r.e4_asr - cl.loc[(r.seed, r.method), "e4_asr"])) for r in df.itertuples()]
df.to_csv(f"{SOUT}/metrics.csv", index=False)
pd.set_option("display.width", 200)
order = ["C", "K3H", "K5", "S3H", "S5", "M3H", "M5"]
print("H (lower is better)"); print(df.pivot_table(index="method", columns="arm", values="H")[order].round(4).to_string())
print("\nMacro-F1 (%)"); print((100 * df.pivot_table(index="method", columns="arm", values="macro_f1"))[order].round(2).to_string())
b = df[df.method == "BRMA"].groupby("arm")[["det_poison", "det_flag"]].sum(); b["detection"] = b.det_flag / b.det_poison
print("\nBRMA detection of poisoned memory after the attack stage"); print(b.round(4).to_string())
def test(arm, a, o):
    x = df[(df.arm == arm) & (df.method == a)].set_index("seed").H; y = df[(df.arm == arm) & (df.method == o)].set_index("seed").H.loc[x.index]
    d = x - y; return x.mean(), y.mean(), int((d < 0).sum()), (wilcoxon(d, alternative="less").pvalue if (d != 0).any() else 1.0)
T = [("T7", "S3H", "BRMA", "NONE"), ("T8", "M3H", "BRMA", "NONE"), ("T9", "K3H", "BRMA", "LOSS5"), ("T10", "S3H", "BRMA", "LOSS5")]
res = [dict(test=t, arm=arm, brma=m1, other=m2, other_method=o, wins=w, p=p) for t, arm, a, o in T for m1, m2, w, p in [test(arm, a, o)]]
rt = pd.DataFrame(res).sort_values("p"); k = len(rt); run_max = 0; adj = []
for i, p in enumerate(rt.p):
    run_max = max(run_max, (k - i) * p); adj.append(min(1, run_max))
rt["p_holm"] = adj; rt = rt.sort_index(); print("\nPre-declared tests"); print(rt.round(5).to_string(index=False))
n_done = len(rows); log(f"results present: {n_done}/672", "(complete)" if n_done >= 672 else "(incomplete: run again to resume)")

In [ ]:
# S-4. Package
import zipfile
zp = f"{W}/brma-adaptive.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{SOUT}/**/*", recursive=True):
        if os.path.isfile(p) and not p.endswith("ownership-source.npz") and not p.endswith("E4-weights.npz"): z.write(p, os.path.relpath(p, W))
log("download:", zp, round(os.path.getsize(zp) / 1e6, 1), "MB")